# End-to-End Credit Card Fraud Detection Benchmark
### A Comparative Study of Traditional Machine Learning, Deep Learning, and Anomaly Detection on Highly Imbalanced Data

---

## Abstract and Project Overview
Credit card fraud represents a critical challenge in financial technology and payment processing systems. This benchmark analyzes, implements, and evaluates multiple machine learning and deep learning methodologies on the ULB European Cardholder Fraud Dataset (284,807 transactions, 492 fraud cases: 0.172% imbalance).

### Methodologies Evaluated:
1. **Traditional Supervised Learning**: Logistic Regression (SMOTE baseline), Random Forest, and XGBoost (scale_pos_weight).
2. **Deep Neural Architectures**: Deep Feedforward ANN with Batch Normalization and Dropout, and an Experimental LSTM sequence model.
3. **Unsupervised Anomaly Detection**: Deep Autoencoder Reconstruction Error modeling on normal transactions.
4. **Hybrid Latent Feature Learning**: Autoencoder Bottleneck Embedding extraction with Random Forest classification.
5. **Model Interpretability and Compliance**: TreeSHAP analysis aligned with GDPR Article 22 explanation requirements.

---
## Section 1: Environment Setup and Dataset Ingestion

In [ ]:
# Environment setup and dependency verification
import os
import sys
import shutil
import glob
import subprocess
import importlib

REQUIRED_PACKAGES = [
    'numpy==1.26.4', 'pandas==2.2.2', 'matplotlib==3.9.0',
    'seaborn==0.13.2', 'scikit-learn==1.5.0', 'tensorflow==2.16.1',
    'xgboost==2.0.3', 'imbalanced-learn==0.12.3', 'kagglehub==0.3.4', 'shap'
]

print("Verifying dependencies...")
for pkg in REQUIRED_PACKAGES:
    name = pkg.split('==')[0]
    mod = name.replace('-', '_')
    try:
        importlib.import_module(mod)
    except Exception:
        try:
            print(f"Installing {pkg}...")
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q', '--disable-pip-version-check'])
        except Exception:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', name, '-q', '--disable-pip-version-check'])

# Automatic download and local caching of the dataset
import kagglehub
dataset_path = kagglehub.dataset_download('mlg-ulb/creditcardfraud')
csv_files = glob.glob(os.path.join(dataset_path, '**', 'creditcard.csv'), recursive=True)

DATASET_DIR = './dataset'
os.makedirs(DATASET_DIR, exist_ok=True)
DATASET_FILE = os.path.join(DATASET_DIR, 'creditcard.csv')

if not os.path.exists(DATASET_FILE) and csv_files:
    shutil.copy(csv_files[0], DATASET_FILE)

print(f"Dataset path: {DATASET_FILE}")
print(f"File size: {os.path.getsize(DATASET_FILE) / (1024 * 1024):.2f} MB")

---
## Section 2: Imports and Global Configuration

In [ ]:
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    roc_curve, precision_recall_curve, average_precision_score,
    f1_score, precision_score, recall_score
)
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model, regularizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
tf.random.set_seed(RANDOM_SEED)

# Visual theme palette
C_LEGIT = '#1982c4'
C_FRAUD = '#ff595e'
C_ACCENT = '#8ac926'
C_WARN = '#ff924c'
C_DARK = '#2b2d42'

PALETTE = {'Legitimate': C_LEGIT, 'Fraud': C_FRAUD}
COLORS = [C_LEGIT, C_FRAUD, C_ACCENT, C_WARN, '#565aa0', '#36949d', '#ffca3a', '#4267ac']

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': '#F8F9FA',
    'axes.edgecolor': '#CCCCCC',
    'axes.labelcolor': C_DARK,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'axes.grid': True,
    'grid.alpha': 0.3,
    'grid.color': '#CCCCCC',
    'xtick.color': C_DARK,
    'ytick.color': C_DARK,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 10
})

sns.set_theme(style='whitegrid', palette=COLORS)

print(f"TensorFlow Version: {tf.__version__}")
print(f"Scikit-Learn Version: {importlib.import_module('sklearn').__version__}")
print(f"Pandas Version: {pd.__version__}")
print(f"Random seed configured: {RANDOM_SEED}")

---
## Section 3: Exploratory Data Analysis (EDA)
Investigating feature properties, distribution skew, and class imbalance.

In [ ]:
# Load dataset
df = pd.read_csv(DATASET_FILE)

print("DATASET OVERVIEW:")
print(f"  Rows: {df.shape[0]:,}")
print(f"  Columns: {df.shape[1]}")
print(f"  Missing values: {df.isnull().sum().sum()}")
print(f"  Duplicate rows: {df.duplicated().sum()}")

class_counts = df['Class'].value_counts()
fraud_pct = (class_counts[1] / len(df)) * 100

print(f"
CLASS BREAKDOWN:")
print(f"  Legitimate (0): {class_counts[0]:,} ({100 - fraud_pct:.4f}%)")
print(f"  Fraud (1)     : {class_counts[1]:,} ({fraud_pct:.4f}%)")
print(f"  Imbalance Ratio: {class_counts[0] // class_counts[1]}:1")

df[['Time', 'Amount', 'Class']].describe().round(4)

In [ ]:
# Class imbalance visual inspection
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle('Class Distribution Analysis', fontsize=14, fontweight='bold')

labels = ['Legitimate (0)', 'Fraud (1)']
counts = [class_counts[0], class_counts[1]]

bars = axes[0].bar(labels, counts, color=[C_LEGIT, C_FRAUD], edgecolor='white', linewidth=1.2, width=0.45)
for bar, count in zip(bars, counts):
    axes[0].text(bar.get_x() + bar.get_width() / 2, bar.get_height() * 1.05,
                 f'{count:,}', ha='center', va='bottom', fontweight='bold')
axes[0].set_title('Absolute Transaction Counts (Log Scale)')
axes[0].set_yscale('log')
axes[0].set_ylabel('Transaction Count')

wedge_props = dict(edgecolor='white', linewidth=2)
axes[1].pie(
    counts, labels=labels, autopct='%1.4f%%',
    colors=[C_LEGIT, C_FRAUD], startangle=140,
    wedgeprops=wedge_props, pctdistance=0.75
)
axes[1].set_title('Proportional Distribution')

plt.tight_layout()
plt.show()

In [ ]:
# Amount and Time distributions partitioned by class
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
fig.suptitle('Transaction Amount and Time Distributions (Fraud vs Legitimate)', fontsize=13, fontweight='bold')

for cls, color, label in [(0, C_LEGIT, 'Legitimate'), (1, C_FRAUD, 'Fraud')]:
    subset = df[df['Class'] == cls]
    axes[0].hist(subset['Amount'], bins=80, alpha=0.6, color=color,
                 label=f'{label} (n={len(subset):,})', density=True)
    axes[1].hist(subset['Time'] / 3600.0, bins=80, alpha=0.6, color=color,
                 label=f'{label}', density=True)

axes[0].set_title('Amount Distribution (Clipped at 500 EUR)')
axes[0].set_xlabel('Amount (EUR)')
axes[0].set_ylabel('Density')
axes[0].set_xlim(0, 500)
axes[0].legend()

axes[1].set_title('Time Distribution (Hours Elapsed)')
axes[1].set_xlabel('Hours since first recorded transaction')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Top discriminative features correlation and KDE
top_corr_features = df.corr()['Class'].abs().sort_values(ascending=False).drop('Class').head(8).index.tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
fig.suptitle('KDE Distributions: Top 8 Discriminative Features', fontsize=13, fontweight='bold')

for ax, feat in zip(axes.flatten(), top_corr_features):
    for cls, color, label in [(0, C_LEGIT, 'Legitimate'), (1, C_FRAUD, 'Fraud')]:
        subset = df[df['Class'] == cls][feat]
        subset.plot.kde(ax=ax, color=color, label=label, linewidth=1.8)
    ax.set_title(feat, fontweight='bold')
    ax.legend(fontsize=8)
    ax.set_xlabel('')

plt.tight_layout()
plt.show()

---
## Section 4: Data Preprocessing and Partitioning
To ensure rigorous evaluation without data leakage:
1. Stratified 3-way partitioning: 80% Training, 10% Validation, 10% Test.
2. StandardScaler is fitted **strictly on the Training set** and applied to Validation and Test.
3. SMOTE oversampling is applied **strictly to the Training set**.

In [ ]:
# Feature and Target separation
X = df.drop(columns=['Class'])
y = df['Class']
feature_names = X.columns.tolist()

# 80/20 train/temp split
X_train_raw, X_temp_raw, y_train, y_temp = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)

# 10/10 val/test split from temp
X_val_raw, X_test_raw, y_val, y_test = train_test_split(
    X_temp_raw, y_temp, test_size=0.50, random_state=RANDOM_SEED, stratify=y_temp
)

# Standardize features fitted strictly on training data
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=feature_names)
X_val_scaled = pd.DataFrame(scaler.transform(X_val_raw), columns=feature_names)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_raw), columns=feature_names)

print(f"Training partition   : {len(X_train_scaled):,} samples | Fraud: {y_train.sum():,}")
print(f"Validation partition : {len(X_val_scaled):,} samples | Fraud: {y_val.sum():,}")
print(f"Test partition       : {len(X_test_scaled):,} samples | Fraud: {y_test.sum():,}")

In [ ]:
# SMOTE oversampling applied strictly to training partition
smote = SMOTE(random_state=RANDOM_SEED, k_neighbors=5)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)

# Calculate class weighting ratio
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
class_weight_ratio = neg_count / pos_count
class_weight_sqrt = np.sqrt(neg_count / pos_count)
class_weights = {0: 1.0, 1: class_weight_sqrt}

print(f"Training set size before SMOTE: {len(X_train_scaled):,} (Fraud: {pos_count:,})")
print(f"Training set size after SMOTE : {len(X_train_smote):,} (Fraud: {(y_train_smote == 1).sum():,})")
print(f"Calculated scale_pos_weight   : {class_weight_ratio:.2f}")
print(f"Adjusted sqrt class weight    : {class_weight_sqrt:.2f}")

---
## Section 5: Standardized Evaluation and Threshold Optimization
Under severe imbalance, default decision thresholds (0.50) are rarely optimal. We implement a validation-tuned threshold optimizer that prevents test set contamination.

In [ ]:
# Evaluation utilities and threshold search
all_results = []

def find_optimal_threshold(y_true_val, y_proba_val, metric='f1'):
    """
    Determines the optimal decision threshold strictly on the validation set.
    Prevents test set contamination and optimistic bias.
    """
    precisions, recalls, thresholds = precision_recall_curve(y_true_val, y_proba_val)
    f1_scores = 2 * (precisions[:-1] * recalls[:-1]) / (precisions[:-1] + recalls[:-1] + 1e-8)
    best_idx = np.argmax(f1_scores)
    best_threshold = thresholds[best_idx]
    return float(best_threshold), float(f1_scores[best_idx])

def evaluate_model(name, y_true, y_pred, y_proba=None):
    """Computes key metrics and stores result dictionary for comparison."""
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    auc = roc_auc_score(y_true, y_proba) if y_proba is not None else np.nan
    ap = average_precision_score(y_true, y_proba) if y_proba is not None else np.nan

    print(f"\n--- Model: {name} ---")
    print(f"  Precision        : {prec:.4f}")
    print(f"  Recall           : {rec:.4f}")
    print(f"  F1-Score         : {f1:.4f}")
    print(f"  ROC-AUC          : {auc:.4f}")
    print(f"  Avg Precision/AP : {ap:.4f}")

    return {
        'Model': name,
        'Precision': round(prec, 4),
        'Recall': round(rec, 4),
        'F1-Score': round(f1, 4),
        'ROC-AUC': round(auc, 4),
        'Avg Precision': round(ap, 4),
        'y_proba': y_proba,
        'y_pred': y_pred
    }

def plot_confusion_matrix(y_true, y_pred, title, ax):
    """Renders styled confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues', ax=ax,
        xticklabels=['Legitimate', 'Fraud'],
        yticklabels=['Legitimate', 'Fraud'],
        linewidths=1, linecolor='white',
        annot_kws={'size': 11, 'fontweight': 'bold'}
    )
    ax.set_title(title, fontweight='bold', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

---
## Section 6: Traditional Machine Learning Models
Benchmarking Logistic Regression, Random Forest, and Gradient Boosted Trees (XGBoost).

In [ ]:
# 6.1 Logistic Regression Baseline (SMOTE balanced)
start_time = time.time()
lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_SEED, solver='lbfgs')
lr.fit(X_train_smote, y_train_smote)

lr_pred = lr.predict(X_test_scaled)
lr_proba = lr.predict_proba(X_test_scaled)[:, 1]

lr_result = evaluate_model('Logistic Regression', y_test, lr_pred, lr_proba)
all_results.append(lr_result)
print(f"Training completed in {time.time() - start_time:.2f}s")

In [ ]:
# 6.2 Random Forest Classifier (SMOTE balanced)
start_time = time.time()
rf = RandomForestClassifier(
    n_estimators=200, max_depth=None, class_weight='balanced',
    random_state=RANDOM_SEED, n_jobs=-1, min_samples_leaf=2
)
rf.fit(X_train_smote, y_train_smote)

rf_pred = rf.predict(X_test_scaled)
rf_proba = rf.predict_proba(X_test_scaled)[:, 1]

rf_result = evaluate_model('Random Forest', y_test, rf_pred, rf_proba)
all_results.append(rf_result)
print(f"Training completed in {time.time() - start_time:.2f}s")

In [ ]:
# 6.3 XGBoost Classifier (Algorithmic class weighting)
start_time = time.time()
xgb = XGBClassifier(
    n_estimators=300, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    scale_pos_weight=class_weight_ratio,
    random_state=RANDOM_SEED, eval_metric='logloss',
    n_jobs=-1
)
xgb.fit(X_train_scaled, y_train)

xgb_pred = xgb.predict(X_test_scaled)
xgb_proba = xgb.predict_proba(X_test_scaled)[:, 1]

xgb_result = evaluate_model('XGBoost', y_test, xgb_pred, xgb_proba)
all_results.append(xgb_result)
print(f"Training completed in {time.time() - start_time:.2f}s")

In [ ]:
# 6.4 Stratified 5-Fold Cross Validation
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)

cv_xgb = cross_val_score(xgb, X_train_scaled, y_train, cv=skf, scoring='roc_auc', n_jobs=-1)
cv_rf = cross_val_score(rf, X_train_smote, y_train_smote, cv=skf, scoring='roc_auc', n_jobs=-1)
cv_lr = cross_val_score(lr, X_train_smote, y_train_smote, cv=skf, scoring='roc_auc', n_jobs=-1)

print(f"XGBoost 5-Fold CV ROC-AUC: {cv_xgb.mean():.4f} (+/- {cv_xgb.std() * 2:.4f})")
print(f"Random Forest CV ROC-AUC: {cv_rf.mean():.4f} (+/- {cv_rf.std() * 2:.4f})")
print(f"Logistic Regression CV ROC-AUC: {cv_lr.mean():.4f} (+/- {cv_lr.std() * 2:.4f})")

---
## Section 7: Explainable AI and Regulatory Compliance (SHAP)
Under GDPR Article 22, automated decision systems must support meaningful explanations. We compute TreeSHAP values for XGBoost.

In [ ]:
import shap

# Initialize TreeExplainer on trained XGBoost model
explainer = shap.TreeExplainer(xgb)
sample_test = X_test_scaled.iloc[:500]
shap_values = explainer.shap_values(sample_test)

plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values, sample_test, feature_names=feature_names, plot_type='dot', show=False)
plt.title('TreeSHAP Global Feature Impact Distribution (XGBoost)', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Individual local explanation for a detected fraud transaction
fraud_indices = np.where(y_test.values == 1)[0]
if len(fraud_indices) > 0:
    target_idx = fraud_indices[0]
    target_sample = X_test_scaled.iloc[target_idx:target_idx+1]
    single_shap = explainer.shap_values(target_sample)[0]

    plt.figure(figsize=(10, 4))
    top_contributors = pd.Series(single_shap, index=feature_names).abs().sort_values(ascending=False).head(8).index
    shap_subset = pd.Series(single_shap, index=feature_names)[top_contributors]

    bars = plt.barh(shap_subset.index, shap_subset.values, color=[C_FRAUD if v > 0 else C_LEGIT for v in shap_subset.values])
    plt.axvline(0, color='black', linestyle='--', linewidth=0.8)
    plt.title(f'Local SHAP Feature Contributions (Test Fraud Sample #{target_idx})', fontsize=11, fontweight='bold')
    plt.xlabel('SHAP Value (Impact on Model Log-Odds)')
    plt.tight_layout()
    plt.show()

---
## Section 8: Deep Learning Architectures
Deep neural networks evaluated with standard early stopping and learning rate scheduling:
1. Artificial Neural Network (Feedforward ANN)
2. Long Short-Term Memory Network (LSTM Sequence Model)
3. Unsupervised Autoencoder (Reconstruction Anomaly Detection)
4. Hybrid Autoencoder Bottleneck + Random Forest Classifier

In [ ]:
# Deep learning data formatting and callback helpers
X_train_np = X_train_scaled.values.astype(np.float32)
y_train_np = y_train.values.astype(np.float32)

X_val_np = X_val_scaled.values.astype(np.float32)
y_val_np = y_val.values.astype(np.float32)

X_test_np = X_test_scaled.values.astype(np.float32)
y_test_np = y_test.values.astype(np.float32)

N_FEATURES = X_train_np.shape[1]

def get_callbacks(monitor='val_loss', patience=6, lr_patience=3):
    return [
        EarlyStopping(monitor=monitor, patience=patience, restore_best_weights=True, verbose=0),
        ReduceLROnPlateau(monitor=monitor, factor=0.5, patience=lr_patience, min_lr=1e-6, verbose=0)
    ]

def plot_training_history(history, model_name):
    fig, axes = plt.subplots(1, 2, figsize=(12, 3.8))
    fig.suptitle(f'{model_name} Training History', fontsize=12, fontweight='bold')

    axes[0].plot(history.history['loss'], label='Train Loss', color=C_LEGIT, linewidth=1.8)
    axes[0].plot(history.history['val_loss'], label='Val Loss', color=C_WARN, linewidth=1.8, linestyle='--')
    axes[0].set_title('Loss Curve')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].legend()

    if 'accuracy' in history.history:
        axes[1].plot(history.history['accuracy'], label='Train Acc', color=C_ACCENT, linewidth=1.8)
        axes[1].plot(history.history['val_accuracy'], label='Val Acc', color=C_FRAUD, linewidth=1.8, linestyle='--')
        axes[1].set_title('Accuracy Curve')
        axes[1].set_xlabel('Epoch')
        axes[1].set_ylabel('Accuracy')
        axes[1].legend()

    plt.tight_layout()
    plt.show()

In [ ]:
# 8.1 Artificial Neural Network (ANN)
def build_ann(n_features, dropout_rate=0.30, learning_rate=1e-3):
    inputs = layers.Input(shape=(n_features,), name='ann_input')
    x = layers.Dense(32, activation='relu')(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(16, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(dropout_rate)(x)
    x = layers.Dense(8, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='ANN')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

ann_model = build_ann(N_FEATURES)
ann_history = ann_model.fit(
    X_train_np, y_train_np,
    epochs=30, batch_size=512,
    validation_data=(X_val_np, y_val_np),
    class_weight=class_weights,
    callbacks=get_callbacks(patience=5),
    verbose=0
)

plot_training_history(ann_history, 'ANN')

# Predict on VALIDATION partition to find optimal threshold (unbiased)
ann_val_proba = ann_model.predict(X_val_np, verbose=0).flatten()
best_ann_threshold, best_val_f1 = find_optimal_threshold(y_val_np, ann_val_proba)
print(f"ANN Optimal Threshold (Validation Tuned): {best_ann_threshold:.4f} (Val F1: {best_val_f1:.4f})")

# Evaluate on TEST partition
ann_test_proba = ann_model.predict(X_test_np, verbose=0).flatten()
ann_test_pred = (ann_test_proba >= best_ann_threshold).astype(int)

ann_result = evaluate_model('ANN', y_test_np, ann_test_pred, ann_test_proba)
all_results.append(ann_result)

In [ ]:
# 8.2 LSTM Sequence Network (Experimental)
X_train_lstm = X_train_np.reshape(-1, N_FEATURES, 1)
X_val_lstm = X_val_np.reshape(-1, N_FEATURES, 1)
X_test_lstm = X_test_np.reshape(-1, N_FEATURES, 1)

def build_lstm(n_timesteps, learning_rate=1e-3):
    inputs = layers.Input(shape=(n_timesteps, 1), name='lstm_input')
    x = layers.LSTM(32, return_sequences=True)(inputs)
    x = layers.Dropout(0.3)(x)
    x = layers.LSTM(16)(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(8, activation='relu')(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = Model(inputs, outputs, name='LSTM')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

lstm_model = build_lstm(N_FEATURES)
lstm_history = lstm_model.fit(
    X_train_lstm, y_train_np,
    epochs=25, batch_size=512,
    validation_data=(X_val_lstm, y_val_np),
    class_weight=class_weights,
    callbacks=get_callbacks(patience=5),
    verbose=0
)

plot_training_history(lstm_history, 'LSTM')

# Validation threshold determination
lstm_val_proba = lstm_model.predict(X_val_lstm, verbose=0).flatten()
best_lstm_threshold, best_val_f1_lstm = find_optimal_threshold(y_val_np, lstm_val_proba)
print(f"LSTM Optimal Threshold (Validation Tuned): {best_lstm_threshold:.4f} (Val F1: {best_val_f1_lstm:.4f})")

# Test evaluation
lstm_test_proba = lstm_model.predict(X_test_lstm, verbose=0).flatten()
lstm_test_pred = (lstm_test_proba >= best_lstm_threshold).astype(int)

lstm_result = evaluate_model('LSTM', y_test_np, lstm_test_pred, lstm_test_proba)
all_results.append(lstm_result)

In [ ]:
# 8.3 Unsupervised Deep Autoencoder (Anomaly Reconstruction)
def build_autoencoder(input_dim):
    input_layer = layers.Input(shape=(input_dim,))
    encoder = layers.Dense(16, activation='relu', activity_regularizer=regularizers.l1(1e-5))(input_layer)
    bottleneck = layers.Dense(8, activation='relu', name='bottleneck')(encoder)
    decoder = layers.Dense(16, activation='relu')(bottleneck)
    output_layer = layers.Dense(input_dim, activation='linear')(decoder)

    autoencoder = Model(inputs=input_layer, outputs=output_layer, name='Autoencoder')
    autoencoder.compile(optimizer='adam', loss='mse')
    return autoencoder

autoencoder = build_autoencoder(N_FEATURES)

# Train exclusively on legitimate transactions
X_train_normal = X_train_scaled.values[y_train == 0]

ae_history = autoencoder.fit(
    X_train_normal, X_train_normal,
    epochs=40, batch_size=512,
    validation_data=(X_val_scaled.values, X_val_scaled.values),
    callbacks=get_callbacks(patience=5),
    verbose=0
)

# Reconstruction error on validation set
val_reconstructions = autoencoder.predict(X_val_scaled.values, verbose=0)
val_mse = np.mean(np.power(X_val_scaled.values - val_reconstructions, 2), axis=1)

best_ae_threshold, best_val_f1_ae = find_optimal_threshold(y_val, val_mse)
print(f"Autoencoder Optimal MSE Threshold (Validation Tuned): {best_ae_threshold:.4f}")

# Apply to test partition
test_reconstructions = autoencoder.predict(X_test_scaled.values, verbose=0)
test_mse = np.mean(np.power(X_test_scaled.values - test_reconstructions, 2), axis=1)
ae_pred = (test_mse > best_ae_threshold).astype(int)

ae_result = evaluate_model('Autoencoder (Anomaly)', y_test, ae_pred, test_mse)
all_results.append(ae_result)

In [ ]:
# 8.4 Hybrid Model: Autoencoder Latent Embeddings + Random Forest
encoder_model = Model(inputs=autoencoder.input, outputs=autoencoder.get_layer('bottleneck').output)

encoded_train = encoder_model.predict(X_train_scaled.values, verbose=0)
encoded_test = encoder_model.predict(X_test_scaled.values, verbose=0)

hybrid_rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=RANDOM_SEED, n_jobs=-1)
hybrid_rf.fit(encoded_train, y_train)

hybrid_pred = hybrid_rf.predict(encoded_test)
hybrid_proba = hybrid_rf.predict_proba(encoded_test)[:, 1]

hybrid_result = evaluate_model('Autoencoder + RF (Hybrid)', y_test, hybrid_pred, hybrid_proba)
all_results.append(hybrid_result)

---
## Section 9: Unified Benchmark Evaluation and Model Comparison

In [ ]:
# Consolidated Results Table
results_df = pd.DataFrame([{
    'Model': r['Model'],
    'Precision': r['Precision'],
    'Recall': r['Recall'],
    'F1-Score': r['F1-Score'],
    'ROC-AUC': r['ROC-AUC'],
    'Avg Precision': r['Avg Precision']
} for r in all_results])

results_df = results_df.sort_values('Avg Precision', ascending=False).reset_index(drop=True)

print("=" * 80)
print("                       CONSOLIDATED MODEL BENCHMARK TABLE")
print("=" * 80)
print(results_df.to_string(index=False))
print("=" * 80)

In [ ]:
# Comparative ROC and Precision-Recall Curves
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, res in enumerate(all_results):
    if res['y_proba'] is not None:
        fpr, tpr, _ = roc_curve(y_test_np, res['y_proba'])
        prec, rec, _ = precision_recall_curve(y_test_np, res['y_proba'])
        axes[0].plot(fpr, tpr, label=f"{res['Model']} (AUC = {res['ROC-AUC']:.3f})", linewidth=1.8, color=COLORS[i % len(COLORS)])
        axes[1].plot(rec, prec, label=f"{res['Model']} (AP = {res['Avg Precision']:.3f})", linewidth=1.8, color=COLORS[i % len(COLORS)])

axes[0].plot([0, 1], [0, 1], 'k--', alpha=0.6, label='Random Baseline (AUC = 0.50)')
axes[0].set_title('ROC Curves Comparison', fontsize=12, fontweight='bold')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate (Recall)')
axes[0].legend(loc='lower right', fontsize=8.5)

baseline_ap = y_test.mean()
axes[1].axhline(y=baseline_ap, color='black', linestyle='--', alpha=0.6, label=f'Random AP ({baseline_ap:.4f})')
axes[1].set_title('Precision-Recall Curves Comparison', fontsize=12, fontweight='bold')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].legend(loc='upper right', fontsize=8.5)

plt.tight_layout()
plt.show()

In [ ]:
# Confusion Matrices across all evaluated models
n_models = len(all_results)
ncols = 3
nrows = (n_models + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(15, 4.5 * nrows))
fig.suptitle('Confusion Matrix Comparison Across All Models', fontsize=14, fontweight='bold')
axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]

for idx, res in enumerate(all_results):
    plot_confusion_matrix(y_test_np, res['y_pred'], res['Model'], axes_flat[idx])

for j in range(n_models, len(axes_flat)):
    axes_flat[j].axis('off')

plt.tight_layout()
plt.show()

---
## Section 10: Key Technical Insights and Production Tradeoffs

### 1. Tabular Inductive Bias: Why Tree Ensembles Win
Gradient Boosted Trees (XGBoost) and Random Forest consistently outperformed deep neural networks on this dataset. Because the 28 principal component features are already decorrelated and orthogonal, the hierarchical feature abstraction capabilities of deep neural networks offer limited advantage over decision trees, which excel at partitioning tabular spaces with irregular decision boundaries.

### 2. Temporal Architectures on Tabular Data
The experimental LSTM model demonstrated that imposing artificial sequential structure onto tabular transaction rows adds parameter overhead without performance benefit. Sequential models should be reserved for genuine longitudinal user transaction histories.

### 3. Unsupervised Anomaly Detection Value
While supervised models achieved higher top-end F1 scores, the Deep Autoencoder provides a critical complementary capability: detecting novel, zero-day fraud patterns without requiring historical fraud labels.

### 4. Regulatory and Business Deployment Tradeoffs
- **Inference Latency**: XGBoost achieves sub-millisecond inference per transaction, fitting real-time payment gateway SLAs (<50ms).
- **Explainability**: TreeSHAP provides exact Shapley values per transaction, satisfying GDPR Article 22 auditability.
- **Cost Sensitivity**: In commercial fraud systems, decision thresholds should be tuned according to a business cost matrix (cost of chargeback fraud vs cost of false positive customer friction) rather than pure F1 optimization.

### References
- Dal Pozzolo, A., Caelen, O., Johnson, R.A. & Bontempi, G. (2015). Calibrating Probability with Undersampling for Unbalanced Classification. IEEE SSCI.
- Lundberg, S.M. & Lee, S.I. (2017). A Unified Approach to Interpreting Model Predictions. NeurIPS.